In [ ]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')

**1. Install useful libraries and U-Net definition**


In [2]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
from pathlib import Path
import cv2
import random
import math

In [ ]:
# Show versioning of deep learning libraries
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Define paths for K-Fold Cross Validation
current_dir = "/content/drive/MyDrive/Colab Notebooks/allnightlong"
k_fold_dir = os.path.join(current_dir, "k10_seed42")
baseline_dir = os.path.join(current_dir)
checkpoint_dir = os.path.join(baseline_dir, "checkpoint_base")

# fold_00 è riservato per il test, quindi usiamo fold_01 a fold_09
available_folds = [f"fold_{i:02d}" for i in range(1, 10)]

print(f"K-Fold directory: {k_fold_dir}")
print(f"Available folds for training/validation: {available_folds}")
print(f"fold_00 riservato per il test")


In [5]:
os.makedirs(checkpoint_dir, exist_ok=True)

**U-NET model**

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """Two consecutive conv-batchnorm-relu layers"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_filters=64, depth=4, bilinear=True):
        super(UNet, self).__init__()
        self.depth = depth
        self.down_layers = nn.ModuleList()
        self.up_layers = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        # Encoder
        filters = init_filters
        for d in range(depth):
            conv = DoubleConv(in_channels, filters)
            self.down_layers.append(conv)
            in_channels = filters
            filters *= 2

        # Bottleneck
        self.bottleneck = DoubleConv(in_channels, filters)

        # Decoder
        for d in range(depth):
            filters //= 2
            if bilinear:
                up = nn.Sequential(
                    nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                    nn.Conv2d(filters * 2, filters, kernel_size=1)
                )
            else:
                up = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)

            conv = DoubleConv(filters * 2, filters)
            self.up_layers.append(nn.ModuleDict({'up': up, 'conv': conv}))

        # Final layer
        self.final_conv = nn.Conv2d(init_filters, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder with skip connections
        skips = []
        for down in self.down_layers:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder with skip connections
        skips = skips[::-1]  # Reverse order
        for i, up_layer in enumerate(self.up_layers):
            x = up_layer['up'](x)
            skip = skips[i]

            # Handle size mismatch
            if x.shape != skip.shape:
                diff_h = skip.shape[2] - x.shape[2]
                diff_w = skip.shape[3] - x.shape[3]
                x = F.pad(x, [diff_w // 2, diff_w - diff_w // 2,
                             diff_h // 2, diff_h - diff_h // 2])

            x = torch.cat([skip, x], dim=1)
            x = up_layer['conv'](x)

        return self.final_conv(x)

**Loss Function - DiceBCELoss**

In [7]:
class DiceBCELoss(nn.Module):
    """Combined Dice + BCE Loss from baseline100"""
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        # BCE component
        bce = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='mean')

        # Dice component
        inputs_prob = torch.sigmoid(inputs)
        inputs_prob = inputs_prob.view(-1)
        targets = targets.view(-1)
        intersection = (inputs_prob * targets).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs_prob.sum() + targets.sum() + smooth)

        return bce + dice_loss

**DSC Calculation Functions**

In [8]:
def calculate_dice_score_single(pred_bin, target):
    """Calcola DSC per una SINGOLA immagine (già binarizzata)."""
    pred_flat = pred_bin.view(-1)
    target_flat = target.view(-1)

    intersection = (pred_flat * target_flat).sum().item()
    pred_sum = pred_flat.sum().item()
    target_sum = target_flat.sum().item()
    union = pred_sum + target_sum

    # DSC Standard (con smoothing)
    smooth = 1e-5
    if union == 0:
        dsc_standard = 1.0
    else:
        dsc_standard = (2. * intersection + smooth) / (union + smooth)

    # DSC Strict (senza smoothing)
    # Se il target è vuoto (maschera nera), restituiamo NaN per ignorarla nella media
    if target_sum == 0:
        dsc_strict = float('nan')
    else:
        dsc_strict = (2. * intersection) / union

    return {'dsc_standard': dsc_standard, 'dsc_strict': dsc_strict}

def calculate_dice_score(pred, target, threshold=0.5):
    """Calcola DSC per un intero batch."""
    if not isinstance(pred, torch.Tensor):
        pred = torch.from_numpy(pred)
    if not isinstance(target, torch.Tensor):
        target = torch.from_numpy(target)

    if pred.min() < 0 or pred.max() > 1:
        pred = torch.sigmoid(pred)

    pred_bin = (pred > threshold).float()

    if pred_bin.dim() == 3:
        pred_bin = pred_bin.unsqueeze(0)
        target = target.unsqueeze(0)

    batch_size = pred_bin.size(0)
    dsc_standard_list = []
    dsc_strict_list = []

    for i in range(batch_size):
        scores = calculate_dice_score_single(pred_bin[i], target[i])
        dsc_standard_list.append(scores['dsc_standard'])
        dsc_strict_list.append(scores['dsc_strict'])

    return {
        'dsc_standard_list': dsc_standard_list,
        'dsc_strict_list': dsc_strict_list
    }

**2. Dataset and Data Loader**

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, ConcatDataset

class LiverDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        # Caricamento immagine originale
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convertiamo in RGB

        # Caricamento maschera
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)

        # Conversione in Tensor (Senza alcuna manipolazione o DA)
        # Portiamo i canali in prima posizione [C, H, W] e normalizziamo a [0, 1]
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask).float().unsqueeze(0)

        return image_tensor, mask_tensor, img_name

def create_fold_datasets(k_fold_dir, train_folds, val_fold):
    """Crea dataset aggregando i fold specificati."""
    def get_paths(fold_name):
        f_path = os.path.join(k_fold_dir, fold_name)
        return os.path.join(f_path, "images"), os.path.join(f_path, "manual_py")

    # --- TRAINING DATASETS ---
    train_datasets_list = []
    for fold_name in train_folds:
        img_dir, msk_dir = get_paths(fold_name)
        if os.path.exists(img_dir) and os.path.exists(msk_dir):
            train_datasets_list.append(LiverDataset(img_dir, msk_dir))
        else:
            print(f" Warning: Fold {fold_name} saltato (cartelle non trovate)")

    if not train_datasets_list:
        raise RuntimeError(f"Nessun fold di training valido trovato in {k_fold_dir}")

    train_dataset = ConcatDataset(train_datasets_list)

    # --- VALIDATION DATASET ---
    val_img, val_msk = get_paths(val_fold)
    if not os.path.exists(val_img):
        raise FileNotFoundError(f"Cartella validation non trovata: {val_img}")

    val_dataset = LiverDataset(val_img, val_msk)

    return train_dataset, val_dataset

In [ ]:
# Dataset creation now handled in training loop for K-Fold rotation
batch_size = 8
print(f"Batch size: {batch_size}")


**3. Training Configuration**

In [ ]:
# Model parameters
init_filters = 64
n_epochs = 50
learning_rate = 1e-4

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model
model = UNet(in_channels=3, out_channels=1, init_filters=init_filters, depth=4, bilinear=True)
model = model.to(device)

# Loss and optimizer
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"\nModel Configuration:")
print(f"  Input channels: 3 (RGB - no preprocessing)")
print(f"  Initial filters: {init_filters}")
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {n_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Loss function: DiceBCELoss")

**4. Save Network Parameters**

In [ ]:
# Save network configuration
config = {
    "model": "UNet",
    "init_filters": init_filters,
    "depth": 4,
    "bilinear": True,
    "batch_size": batch_size,
    "n_epochs": n_epochs,
    "learning_rate": learning_rate,
    "loss_function": "DiceBCELoss",
    "optimizer": "Adam",
    "scheduler": "ReduceLROnPlateau",
    "in_channels": 3,
    "preprocessing": "none - RGB original images"
}

config_path = os.path.join(checkpoint_dir, 'network_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)

print(f"Configuration saved to: {config_path}")

**5. Training Loop**

In [ ]:
# --- CONFIGURAZIONE FISSA DEI FOLD ---
val_fold = "fold_05"
available_folds = [f"fold_{i:02d}" for i in range(1, 10)]
train_folds = [f for f in available_folds if f != val_fold] # Esclude fold_05 dal training

# Parametri e stato
threshold = 0.5
best_val_dice_standard = 0.0
early_stopping_patience = 30
patience_counter = 0
start_epoch = 0

print("="*60)
print(f"START TRAINING (Validation: {val_fold} | Training: {train_folds})")
print(f"Salvataggio in: {checkpoint_dir}")
print("="*60)

for epoch in range(start_epoch, n_epochs):
    print(f"\nEPOCH {epoch+1}/{n_epochs}")

    # Caricamento dati (Fold 05 escluso dal training)
    # --- RIGA CORRETTA DA USARE NELLA CELLA 16 ---
    train_dataset, val_dataset = create_fold_datasets(k_fold_dir, train_folds, val_fold)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # --- PHASE: TRAINING ---
    model.train()
    train_loss = 0.0
    for images, masks, names in tqdm(train_loader, desc="Training"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks) # Utilizza DiceBCELoss
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)

    # --- PHASE: VALIDATION ---
    model.eval()
    val_dice_standard_list, val_dice_strict_list = [], []
    all_probs, all_names = [], []

    with torch.no_grad():
        for images, masks, names in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)

            res = calculate_dice_score(outputs, masks, threshold=threshold) #
            val_dice_standard_list.extend(res['dsc_standard_list'])
            val_dice_strict_list.extend(res['dsc_strict_list'])
            all_probs.append(torch.sigmoid(outputs).cpu().numpy())
            all_names.extend(names)

    # Calcolo metriche
    train_loss /= len(train_loader.dataset)
    dsc_standard = np.mean(val_dice_standard_list)
    dsc_strict = np.nanmean(val_dice_strict_list) if not np.all(np.isnan(val_dice_strict_list)) else 0.0

    print(f"  Results: Loss: {train_loss:.4f} | DSC Std: {dsc_standard:.4f} | DSC Strict: {dsc_strict:.4f}")

    # Logica di salvataggio record su DSC Standard
    if dsc_standard > best_val_dice_standard:
        print(f" NUOVO RECORD! {best_val_dice_standard:.4f} -> {dsc_standard:.4f}")
        best_val_dice_standard = dsc_standard
        patience_counter = 0

        # Salvataggio modello
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'dsc_standard': dsc_standard,
            'val_fold': val_fold
        }, os.path.join(checkpoint_dir, 'best_model.pth'))

        # Salvataggio maschere predette
        results_dir = os.path.join(checkpoint_dir, 'best_validation_masks')
        os.makedirs(results_dir, exist_ok=True)
        y_pred_all = (np.concatenate(all_probs) > threshold).astype(np.uint8)
        for i in range(len(all_names)):
            cv2.imwrite(os.path.join(results_dir, f"{os.path.splitext(all_names[i])[0]}_pred.png"), y_pred_all[i].squeeze() * 255)
    else:
        patience_counter += 1
        print(f"  Nessun miglioramento. Patience: {patience_counter}/{early_stopping_patience}")

    if patience_counter >= early_stopping_patience:
        print("\nEarly stopping raggiunto.")
        break